In [1]:
import os
import re
import json
import unicodedata
from datetime import datetime

import pandas as pd

# =========================
# CẤU HÌNH ĐƯỜNG DẪN
# =========================
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("D:\\TTTN\\Project\\scripts\\preprocessing.ipynb")))
RAW_INPUT_PATH = os.path.join(BASE_DIR, "data_topcv", "topcv_all_fields_merged_2026-03-16_20-57-23.xlsx")
OUTPUT_DIR = os.path.join(BASE_DIR, "data_processed")

print("BASE_DIR         :", BASE_DIR)
print("RAW_INPUT_PATH   :", RAW_INPUT_PATH)
print("OUTPUT_DIR       :", OUTPUT_DIR)
print("File tồn tại?"   , os.path.exists(RAW_INPUT_PATH))

BASE_DIR         : D:\TTTN\Project
RAW_INPUT_PATH   : D:\TTTN\Project\data_topcv\topcv_all_fields_merged_2026-03-16_20-57-23.xlsx
OUTPUT_DIR       : D:\TTTN\Project\data_processed
File tồn tại? True


In [2]:
# pd.set_option('display.max_columns', None)   # hiển thị tất cả cột
# pd.set_option('display.max_rows', None)      # hiển thị tất cả dòng
# pd.set_option('display.width', None)         # không giới hạn chiều rộng
# pd.set_option('display.max_colwidth', None)  # không cắt nội dung cột

In [3]:
# Dictionary mapping skill
SKILL_DICT = {
    "python": ["python"],
    "sql": ["sql", "mysql", "postgresql", "sql server", "mssql"],
    "excel": ["excel", "microsoft excel"],
    "power_bi": ["power bi", "powerbi"],
    "tableau": ["tableau"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "scikit_learn": ["scikit-learn", "sklearn"],
    "machine_learning": ["machine learning", "ml"],
    "deep_learning": ["deep learning", "dl"],
    "spark": ["spark", "apache spark"],
    "hadoop": ["hadoop"],
    "airflow": ["airflow"],
    "docker": ["docker"],
    "git": ["git", "github", "gitlab"],
    "statistics": ["thống kê", "statistics", "statistical"],
    "data_visualization": ["trực quan hóa dữ liệu", "data visualization", "visualization"],
}

print("Số skill trong dict:", len(SKILL_DICT))
print("Các skill:", list(SKILL_DICT.keys()))

Số skill trong dict: 17
Các skill: ['python', 'sql', 'excel', 'power_bi', 'tableau', 'pandas', 'numpy', 'scikit_learn', 'machine_learning', 'deep_learning', 'spark', 'hadoop', 'airflow', 'docker', 'git', 'statistics', 'data_visualization']


# 1. Clean cơ bản

In [4]:
# Chuẩn hóa giá trị trống
def normalize_empty_value(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if not val_str or val_str.lower() in {"nan", "none", "null"}:
        return None
    return val_str

# Lấy giá trị đầu tiên không rỗng từ danh sách
def first_non_empty(*values):
    for v in values:
        v = normalize_empty_value(v)
        if v is not None:
            return v
    return None

# Chuẩn hóa unicode
def normalize_unicode(text: str) -> str:
    if text is None:
        return ""
    return unicodedata.normalize("NFC", str(text))

# Làm sạch văn bản
def clean_text(text: str) -> str:
    text = normalize_empty_value(text)
    if text is None:
        return ""
    
    text = normalize_unicode(text)                 # chuẩn hóa unicode
    text = text.lower()                            # chuyển về chữ thường
    text = re.sub(r"<[^>]+>", " ", text)           # bỏ html
    text = re.sub(r"[\r\n\t]+", " ", text)         # thay thế ký tự xuống dòng, tab bằng dấu cách
    text = text.replace("−", ". ").replace("•", ". ").replace("▪", ". ").replace("✅", ". ") # thay thế ký tự đặc biệt bằng dấu cách
    text = re.sub(r'[^a-zA-Z0-9à-ỹ\s\.,]', ' ', text)  # loại bỏ ký tự đặc biệt, chỉ giữ lại chữ, số, dấu chấm và khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()       # loại bỏ khoảng trắng thừa
    return text

print("Các hàm hỗ trợ đã định nghĩa.")

Các hàm hỗ trợ đã định nghĩa.


In [5]:
# Load dữ liệu thô
def load_raw_data(input_path: str) -> pd.DataFrame:
    ext = os.path.splitext(input_path)[1].lower()
    if ext == ".xlsx":
        df = pd.read_excel(input_path)
    elif ext == ".csv":
        df = pd.read_csv(input_path)
    else:
        raise ValueError(f"Định dạng file không hỗ trợ: {ext}")
    
    print(f"[INFO] Loaded raw data: {df.shape[0]} rows x {df.shape[1]} cols")
    return df


# Thực thi
raw_df = load_raw_data(RAW_INPUT_PATH)

# Kiểm tra nhanh
display(raw_df.head(3))
print("\nColumns:\n", raw_df.columns.tolist())

[INFO] Loaded raw data: 325 rows x 33 cols


,job_url,source_field_name,field_count,title,detail_title,company_name,company_name_full,company_url,company_url_from_job,salary_list,...,desc_mota,desc_yeucau,desc_quyenloi,company_website,company_scale_from_job,company_scale,company_field_from_job,company_address_from_job,company_address,company_description
0,https://www.topcv.vn/brand/beeogisticscorporat...,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,Junior FP&A Analyst - Hồ Chí Minh,Bee Logistics Corporation,Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporat...,https://www.topcv.vn/brand/beeogisticscorporat...,12 - 20 triệu,...,"– Overseeing all financial planning, reporting...",– At least 2 year’ experience in fthe inance/a...,– Competitive salary according to personal exp...,http://www.beelogistics.com/,NaN,500-1000,NaN,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu...",Xuất phát với ước mơ xây dựng một doanh nghiệp...
1,https://www.topcv.vn/brand/educa/tuyen-dung/da...,Data Analyst,1,Data Governance Specialist,Data Governance Specialist,EDUPIA,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://www.topcv.vn/brand/educa?id=17270,20 - 30 triệu,...,"− Xây dựng, triển khai và duy trì khung Data G...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên...",Mức lương thỏa thuận theo năng lực. Thử việc h...,https://edupia.vn/,NaN,500-1000,NaN,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/pr...,AI Engineer,1,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,EDUPIA,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://www.topcv.vn/brand/educa?id=17270,30 - 35 triệu,...,1. Lập kế hoạch & Xây dựng chiến lược AI (20%)...,Tối thiểu 2–3 năm kinh nghiệm ở vị trí Product...,Tinh thần: - Làm việc trong môi trường start-u...,https://edupia.vn/,NaN,500-1000,NaN,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."



Columns:
 ['job_url', 'source_field_name', 'field_count', 'title', 'detail_title', 'company_name', 'company_name_full', 'company_url', 'company_url_from_job', 'salary_list', 'detail_salary', 'address_list', 'detail_location', 'exp_list', 'detail_experience', 'deadline', 'tags', 'job_level', 'education_level', 'job_quantity', 'employment_type', 'working_addresses', 'working_times', 'desc_mota', 'desc_yeucau', 'desc_quyenloi', 'company_website', 'company_scale_from_job', 'company_scale', 'company_field_from_job', 'company_address_from_job', 'company_address', 'company_description']


In [6]:
# Gộp cột đại diện
def merge_semantic_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["job_url"] = df.get("job_url")
    out["source_field_name"] = df.get("source_field_name")
    out["field_count"] = df.get("field_count")

    out["job_title_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("detail_title"), df.get("title"))]
    
    out["salary_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("detail_salary"), df.get("salary_list"))]
    out["location_raw"] = [first_non_empty(a, b) for a,b in zip(df.get("detail_location"), df.get("address_list"))]
    out["working_addresses_raw"] = df.get("working_addresses")
    out["working_times_raw"] = df.get("working_times")
    out["experience_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("detail_experience"), df.get("exp_list"))]
    out["tags_raw"] = df.get("tags")
    
    out["job_level_raw"] = df.get("job_level")
    out["education_level_raw"] = df.get("education_level")
    out["employment_type_raw"] = df.get("employment_type")
    out["job_quantity"] = df.get("job_quantity")
    out["deadline_raw"] = df.get("deadline")
    
    out["description_raw"]  = df.get("desc_mota")
    out["requirements_raw"] = df.get("desc_yeucau")
    out["benefits_raw"]     = df.get("desc_quyenloi")

    out["company_name_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("company_name_full"), df.get("company_name"))]
    out["company_url"] = [first_non_empty(a, b) for a, b in zip(df.get("company_url_from_job"), df.get("company_url"))]
    out["company_website_raw"] = df.get("company_website")
    out["company_scale_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("company_scale_from_job"), df.get("company_scale"))]
    out["company_field_raw"] = df.get("company_field_from_job")
    out["company_address_raw"] = [first_non_empty(a, b) for a, b in zip(df.get("company_address_from_job"), df.get("company_address"))]
    out["company_description_raw"] = df.get("company_description")

    print(f"[INFO] After merging: {out.shape[0]} rows x {out.shape[1]} cols")
    return out


# Chạy
df = merge_semantic_columns(raw_df)
display(df.head(3))
df.info()

[INFO] After merging: 325 rows x 25 cols


,job_url,source_field_name,field_count,job_title_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,tags_raw,...,description_raw,requirements_raw,benefits_raw,company_name_raw,company_url,company_website_raw,company_scale_raw,company_field_raw,company_address_raw,company_description_raw
0,https://www.topcv.vn/brand/beeogisticscorporat...,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,12 - 20 triệu,Hồ Chí Minh,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn Data Analyst; Tài chính; Kế toán,...,"– Overseeing all financial planning, reporting...",– At least 2 year’ experience in fthe inance/a...,– Competitive salary according to personal exp...,Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporat...,http://www.beelogistics.com/,500-1000,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu...",Xuất phát với ước mơ xây dựng một doanh nghiệp...
1,https://www.topcv.vn/brand/educa/tuyen-dung/da...,Data Analyst,1,Data Governance Specialist,20 - 30 triệu,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn Data Analyst,...,"− Xây dựng, triển khai và duy trì khung Data G...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên...",Mức lương thỏa thuận theo năng lực. Thử việc h...,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://edupia.vn/,500-1000,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/pr...,AI Engineer,1,Project Manager Dự Án AI HUB,30 - 35 triệu,Hà Nội,(đã được cập nhật theo Danh mục Hành chính mới...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 0...,2 năm,Chuyên môn IT Project Manager,...,1. Lập kế hoạch & Xây dựng chiến lược AI (20%)...,Tối thiểu 2–3 năm kinh nghiệm ở vị trí Product...,Tinh thần: - Làm việc trong môi trường start-u...,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://edupia.vn/,500-1000,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Towe...","Được thành lập năm 2018, Edupia là Edtech lớn ..."


<class 'pandas.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   job_url                  325 non-null    str  
 1   source_field_name        325 non-null    str  
 2   field_count              325 non-null    int64
 3   job_title_raw            325 non-null    str  
 4   salary_raw               325 non-null    str  
 5   location_raw             325 non-null    str  
 6   working_addresses_raw    325 non-null    str  
 7   working_times_raw        294 non-null    str  
 8   experience_raw           325 non-null    str  
 9   tags_raw                 325 non-null    str  
 10  job_level_raw            325 non-null    str  
 11  education_level_raw      325 non-null    str  
 12  employment_type_raw      325 non-null    str  
 13  job_quantity             325 non-null    str  
 14  deadline_raw             325 non-null    str  
 15  description_raw  

In [7]:
df_clean = df.copy()

In [ ]:
# Làm sạch các cột văn bản (lowercase, bỏ html, bỏ ký tự đặc biệt, loại bỏ khoảng trắng thừa)
clean_columns = {
    "job_title_raw": "job_title_raw",
    "location_raw": "location_raw",
    # "working_addresses_raw": "working_addresses_raw",
    "tags_raw": "tags_raw",
    "description_raw": "description_raw",
    "requirements_raw": "requirements_raw",
    "benefits_raw": "benefits_raw",
    "company_field_raw": "company_field_raw",
    "company_description_raw": "company_description_raw"
}

for raw_col, clean_col in clean_columns.items():
    df_clean[clean_col] = df_clean[raw_col].apply(clean_text)

print("Đã tạo df_clean và clean xong các cột!")
display(df_clean[list(clean_columns.values())].head(8))

Đã tạo df_clean và clean xong các cột!


,job_title_raw,location_raw,tags_raw,description_raw,requirements_raw,benefits_raw,company_field_raw,company_description_raw
0,junior fp a analyst hồ chí minh,hồ chí minh,chuyên môn data analyst tài chính kế toán,"overseeing all financial planning, reporting a...",at least 2 year experience in fthe inance acco...,competitive salary according to personal exper...,,xuất phát với ước mơ xây dựng một doanh nghiệp...
1,data governance specialist,hà nội,chuyên môn data analyst,". xây dựng, triển khai và duy trì khung data g...",". đã tốt nghiệp đh trở lên, ưu tiên các chuyên...",mức lương thỏa thuận theo năng lực. thử việc h...,,"được thành lập năm 2018, edupia là edtech lớn ..."
2,project manager dự án ai hub,hà nội,chuyên môn it project manager,1. lập kế hoạch xây dựng chiến lược ai 20 xây ...,tối thiểu 2 3 năm kinh nghiệm ở vị trí product...,tinh thần làm việc trong môi trường start up c...,,"được thành lập năm 2018, edupia là edtech lớn ..."
3,ai engineer,hà nội,chuyên môn ai engineer it phần mềm,xây dựng các capability ai và data processing ...,kinh nghiệm ai ml ứng dụng thực tế python và x...,"làm ai gắn với hệ thống thật, không làm resear...",,"fpt is là công ty cung cấp sản phẩm, giải pháp..."
4,ai engineer,hà nội,chuyên môn ai engineer,nghiên cứu và triển khai các giải pháp công ng...,"tốt nghiệp đại học chuyên ngành cntt, khoa học...",mức lương cạnh tranh tham gia bảo hiểm đầy đủ ...,,"fpt is là công ty cung cấp sản phẩm, giải pháp..."
5,data analyst,hà nội,chuyên môn data analyst,"tham gia các các dự án về quản lý, tích hợp và...",trên 3 năm kinh nghiệm ở vị trí data analyst c...,thu nhập cạnh tranh theo năng lực ứng viên. lư...,,"fpt is là công ty cung cấp sản phẩm, giải pháp..."
6,data engineer,hà nội,chuyên môn data engineer,"triển khai tích hợp dữ liệu, xử lý dữ liệu trê...",tốt nghiệp đại học trở lên các chuyên ngành cô...,thu nhập cạnh tranh theo năng lực ứng viên. lư...,,"fpt is là công ty cung cấp sản phẩm, giải pháp..."
7,fresher data engineer,hà nội,chuyên môn data engineer it phần mềm,data scientist the sexiest job of the 21st cen...,là sinh viên năm cuối đã tốt nghiệp đại học ch...,được nhận trợ cấp đào tạo toàn khóa học lên đế...,,fpt software công ty thành viên của tập đoàn f...


# 2. Chuẩn hóa các cột

## - Extract khu vực

In [ ]:
# =========================
# DANH SÁCH THÀNH PHỐ / TỈNH CHUẨN
# =========================
VIETNAM_CITIES = {
    # TP. Hà Nội
    "hà nội", "ha noi", "tp hà nội", "tp ha noi",
    # TP. Hồ Chí Minh
    "hồ chí minh", "ho chi minh", "tp hồ chí minh", "tp ho chi minh",
    # TP. Hải Phòng
    "hải phòng", "hai phong", "tp hải phòng", "tp hai phong",
    # TP. Huế
    "Huế", "hue", "tp huế", "tp hue",
    # TP. Đà Nẵng
    "đà nẵng", "da nang", "tp đà nẵng", "tp da nang",
    # TP. Cần Thơ
    "cần thơ", "can tho", "tp cần thơ", "tp can tho",
    
    "an giang", "an giang",
    "bắc ninh", "bac ninh",
    "cà mau", "ca mau",
    "cao bằng", "cao bang",
    "đắk lắk", "dak lak",
    "điện biên", "dien bien",
    "đồng nai", "dong nai",
    "đồng tháp", "dong thap",
    "gia lai", "gia lai",
    "hà tĩnh", "ha tinh",
    "hưng yên", "hung yen",
    "khánh hòa", "khanh hoa",
    "lai châu", "lai chau",
    "lâm đồng", "lam dong",
    "lạng sơn", "lang son",
    "lào cai", "lao cai",
    "nghệ an", "nghe an",
    "ninh bình", "ninh binh",
    "phú thọ", "phu tho",
    "quảng ngãi", "quang ngai",
    "quảng ninh", "quang ninh"
    "quảng trị", "quang tri",
    "sơn la", "son la",
    "tây ninh", "tay ninh",
    "thái nguyên", "thai nguyen",
    "thanh hóa", "thanh hoa",
    "tuyên quang", "tuyen quang",
    "vĩnh long", "vinh long",
}

# =========================
# HÀM TẠO clean location
# =========================
def clean_location(location_raw: str, working_addresses_raw: str) -> str:
    if pd.isna(location_raw) or not str(location_raw).strip():
        return ""
    
    loc = str(location_raw).strip()
    
    # Kiểm tra có nhiều địa điểm không
    multi_keywords = {"và", "&", "or", ",", "với", "cùng", "và các"}
    has_multiple = any(kw in loc.lower() for kw in multi_keywords)
    
    # Trường hợp chỉ 1 địa điểm → trả về nguyên bản
    if not has_multiple:
        return loc
    
    # Trường hợp nhiều địa điểm → trích xuất tất cả từ working_addresses_raw
    if pd.isna(working_addresses_raw) or not str(working_addresses_raw).strip():
        return loc  # fallback
    
    addr = str(working_addresses_raw).lower()   # so sánh không phân biệt hoa thường
    
    found_cities = []
    for city in VIETNAM_CITIES:
        if city in addr:
            # Lấy phiên bản có dấu đẹp để hiển thị
            found_cities.append(city)
    
    if found_cities:
        # Loại bỏ trùng lặp và sắp xếp theo thứ tự xuất hiện trong danh sách gốc
        seen = set()
        unique_cities = []
        for city in found_cities:
            if city not in seen:
                seen.add(city)
                unique_cities.append(city)
        
        return ", ".join(unique_cities)
    
    # Nếu không tìm thấy thành phố nào trong danh sách
    return loc

df_clean["location_clean"] = df_clean.apply(
    lambda row: clean_location(
        row.get("location_raw", ""), 
        row.get("working_addresses_raw", "") or row.get("working_addresses_clean", "")
    ), 
    axis=1
)

In [ ]:
# Kiểm tra kết quả
print("=== Kết quả location_clean ===")
print(f"Tổng số giá trị duy nhất: {df_clean['location_clean'].nunique()}\n")

display(df_clean[["location_raw", "working_addresses_raw", "location_clean"]].head(15))

# Xem các trường hợp có nhiều địa điểm
multi = df_clean[df_clean["location_raw"].str.contains("và|&|or|,", na=False, case=False)]
print(f"\nSố job có nhiều địa điểm: {len(multi)}")
if len(multi) > 0:
    display(multi[["location_raw", "location_clean"]].head(10))

=== Kết quả location_clean ===
Tổng số giá trị duy nhất: 14



,location_raw,working_addresses_raw,location_clean
0,hồ chí minh,(đã được cập nhật theo Danh mục Hành chính mới...,hồ chí minh
1,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
2,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
3,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
4,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
5,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
6,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
7,hà nội,(đã được cập nhật theo Danh mục Hành chính mới...,hà nội
8,hồ chí minh,(đã được cập nhật theo Danh mục Hành chính mới...,hồ chí minh
9,hồ chí minh,(đã được cập nhật theo Danh mục Hành chính mới...,hồ chí minh



Số job có nhiều địa điểm: 9


,location_raw,location_clean
48,"hà nội , hồ chí minh","hồ chí minh, hà nội"
63,hà nội và 18 nơi khác,"đắk lắk, đà nẵng, hồ chí minh, lâm đồng, vĩnh ..."
64,"hà nội , hồ chí minh","hồ chí minh, hà nội"
115,"hà nội , phú thọ","hà nội, phú thọ"
181,"hà nội , hồ chí minh","hồ chí minh, hà nội"
295,hồ chí minh và 2 nơi khác,"đà nẵng, hồ chí minh, hà nội"
306,hồ chí minh và 3 nơi khác,"đà nẵng, hồ chí minh, khánh hòa, hà nội"
309,"hà nội , hồ chí minh","hồ chí minh, hà nội"
319,"hà nội , phú thọ","hà nội, phú thọ"


In [ ]:
pd.set_option('display.max_columns', None)   # hiển thị tất cả cột
pd.set_option('display.max_rows', None)      # hiển thị tất cả dòng
pd.set_option('display.width', None)         # không giới hạn chiều rộng
pd.set_option('display.max_colwidth', None)  # không cắt nội dung cột

In [12]:
pd.set_option('display.max_colwidth', None)  # không cắt nội dung cột

- Chuẩn hóa địa chỉ làm việc (working_addresses)

In [22]:
def clean_working_addresses_solution_1(raw_text, vietnam_cities):
    import re
    import unicodedata
    import pandas as pd

    WARD_PATTERN = re.compile(
        r'(?i)\b(phường|xã|thị trấn)\s+[^\n,;()]+'
    )

    DISTRICT_PATTERN = re.compile(
        r'(?i)\b(quận|huyện|thị xã|thành phố)\s+[^\n,;()]+'
    )

    FLOOR_PATTERN = re.compile(
        r'(?i)\b(?:tầng|lầu|floor|fl)\s*[a-z0-9,\-–/ ]+'
    )

    BUILDING_HINT_PATTERN = re.compile(
        r'(?i)\b(?:tòa|toà|tower|building|plaza|center|centre|complex|landmark|office)\b'
    )

    STREET_PATTERN = re.compile(
        r'(?i)\b(?:số\s*)?\d+[a-zA-Z0-9/\-]*\s+[^,]+'
    )

    def normalize_empty_value(val):
        if pd.isna(val):
            return None
        val_str = str(val).strip()
        if not val_str or val_str.lower() in {"nan", "none", "null"}:
            return None
        return val_str

    def normalize_unicode(text):
        if text is None:
            return ""
        return unicodedata.normalize("NFC", str(text))

    def clean_address_text(text):
        text = normalize_empty_value(text)
        if text is None:
            return ""
        text = normalize_unicode(text)
        text = text.replace("\\n", "\n")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\s*\n\s*", "\n", text)
        return text.strip()

    def remove_leading_note(text):
        text = clean_address_text(text)
        if not text:
            return ""
        return re.sub(r'^\s*\([^)]*\)\s*', '', text).strip()

    def detect_city_from_text(text):
        text = clean_address_text(text)
        if not text:
            return None
        text_lower = text.lower()
        for city in vietnam_cities:
            city_norm = normalize_empty_value(city)
            if city_norm and city_norm.lower() in text_lower:
                return city_norm
        return None

    def extract_district_note(text):
        text = clean_address_text(text)
        if not text:
            return None
        matches = re.findall(r'\(([^)]*)\)', text)
        matches = [m.strip() for m in matches if normalize_empty_value(m)]
        return "; ".join(matches) if matches else None

    def remove_parentheses_content(text):
        text = clean_address_text(text)
        if not text:
            return ""
        return re.sub(r'\([^)]*\)', '', text).strip()

    def parse_location_detail_layered(location_detail):
        if not location_detail:
            return {
                "floor": None,
                "building": None,
                "street_address": None
            }

        chunks = [c.strip() for c in location_detail.split(",") if c.strip()]
        floor = None
        building = None
        street_address = None
        leftovers = []

        for chunk in chunks:
            floor_match = FLOOR_PATTERN.search(chunk)
            street_match = STREET_PATTERN.search(chunk)
            building_hint = BUILDING_HINT_PATTERN.search(chunk)

            if floor_match:
                floor = floor_match.group(0).strip()
                rest = FLOOR_PATTERN.sub("", chunk).strip(" ,-")
                if rest:
                    building = f"{building}, {rest}" if building else rest
                continue

            if street_match and street_match.group(0).strip() == chunk:
                street_address = chunk
                continue

            if building_hint:
                building = f"{building}, {chunk}" if building else chunk
                continue

            leftovers.append(chunk)

        if leftovers:
            extra = ", ".join(leftovers)
            building = f"{building}, {extra}" if building else extra

        return {
            "floor": floor,
            "building": building,
            "street_address": street_address
        }

    def empty_result():
        return {
            "city": None,
            "ward": None,
            "district": None,
            "district_note": None,
            "floor": None,
            "building": None,
            "street_address": None,
            "location_detail": None,
            "full_parsed": None,
            "parse_method": "solution_1"
        }

    raw_text = normalize_empty_value(raw_text)
    if raw_text is None:
        return empty_result()

    text = clean_address_text(raw_text)
    text = remove_leading_note(text)
    text = re.sub(r'^\-\s*', '', text).strip()

    if not text:
        return empty_result()

    city = None
    detail = text

    if ":" in text:
        left, right = text.split(":", 1)
        possible_city = left.strip()
        detail = right.strip()
        city = detect_city_from_text(possible_city)
        if not city:
            city = detect_city_from_text(text)
    else:
        city = detect_city_from_text(text)

    district_note = extract_district_note(detail)
    detail_no_note = remove_parentheses_content(detail)

    ward_match = WARD_PATTERN.search(detail_no_note)
    ward = ward_match.group(0).strip() if ward_match else None

    district_match = DISTRICT_PATTERN.search(detail_no_note)
    district = district_match.group(0).strip() if district_match else None

    location_detail = detail_no_note
    if ward_match:
        location_detail = detail_no_note[:ward_match.start()].strip(" ,")

    layered = parse_location_detail_layered(location_detail)

    full_parts = [
        layered.get("street_address") or layered.get("building") or location_detail,
        ward,
        city
    ]
    full_parsed = ", ".join([p for p in full_parts if p]) if any(full_parts) else None

    return {
        "city": city,
        "ward": ward,
        "district": district,
        "district_note": district_note,
        "floor": layered.get("floor"),
        "building": layered.get("building"),
        "street_address": layered.get("street_address"),
        "location_detail": location_detail if location_detail else None,
        "full_parsed": full_parsed,
        "parse_method": "solution_1"
    }

In [23]:
def clean_working_addresses_solution_2(raw_text, vietnam_cities):
    import re
    import unicodedata
    import pandas as pd

    WARD_PATTERN = re.compile(
        r'(?i)\b(phường|xã|thị trấn)\s+[^\n,;()]+'
    )

    DISTRICT_PATTERN = re.compile(
        r'(?i)\b(quận|huyện|thị xã|thành phố)\s+[^\n,;()]+'
    )

    FLOOR_PATTERN = re.compile(
        r'(?i)\b(?:tầng|lầu|floor|fl)\s*[a-z0-9,\-–/ ]+'
    )

    BUILDING_PATTERN = re.compile(
        r'(?i)(?:\b(?:tòa|toà)\s+[^\n,;()]+|\b[^\n,;()]*\b(?:tower|building|plaza|center|centre|complex|landmark|office)\b[^\n,;()]*)'
    )

    STREET_PATTERN = re.compile(
        r'(?i)\b(?:số\s*)?\d+[a-zA-Z0-9/\-]*\s+[^,]+'
    )

    def normalize_empty_value(val):
        if pd.isna(val):
            return None
        val_str = str(val).strip()
        if not val_str or val_str.lower() in {"nan", "none", "null"}:
            return None
        return val_str

    def normalize_unicode(text):
        if text is None:
            return ""
        return unicodedata.normalize("NFC", str(text))

    def clean_address_text(text):
        text = normalize_empty_value(text)
        if text is None:
            return ""
        text = normalize_unicode(text)
        text = text.replace("\\n", "\n")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\s*\n\s*", "\n", text)
        return text.strip()

    def remove_leading_note(text):
        text = clean_address_text(text)
        if not text:
            return ""
        return re.sub(r'^\s*\([^)]*\)\s*', '', text).strip()

    def detect_city_from_text(text):
        text = clean_address_text(text)
        if not text:
            return None
        text_lower = text.lower()
        for city in vietnam_cities:
            city_norm = normalize_empty_value(city)
            if city_norm and city_norm.lower() in text_lower:
                return city_norm
        return None

    def extract_district_note(text):
        text = clean_address_text(text)
        if not text:
            return None
        matches = re.findall(r'\(([^)]*)\)', text)
        matches = [m.strip() for m in matches if normalize_empty_value(m)]
        return "; ".join(matches) if matches else None

    def remove_parentheses_content(text):
        text = clean_address_text(text)
        if not text:
            return ""
        return re.sub(r'\([^)]*\)', '', text).strip()

    def empty_result():
        return {
            "city": None,
            "ward": None,
            "district": None,
            "district_note": None,
            "floor": None,
            "building": None,
            "street_address": None,
            "location_detail": None,
            "full_parsed": None,
            "parse_method": "solution_2"
        }

    raw_text = normalize_empty_value(raw_text)
    if raw_text is None:
        return empty_result()

    text = clean_address_text(raw_text)
    text = remove_leading_note(text)
    text = re.sub(r'^\-\s*', '', text).strip()

    if not text:
        return empty_result()

    city = detect_city_from_text(text)
    district_note = extract_district_note(text)
    text_no_note = remove_parentheses_content(text)

    ward_match = WARD_PATTERN.search(text_no_note)
    ward = ward_match.group(0).strip() if ward_match else None

    district_match = DISTRICT_PATTERN.search(text_no_note)
    district = district_match.group(0).strip() if district_match else None

    floor_match = FLOOR_PATTERN.search(text_no_note)
    floor = floor_match.group(0).strip() if floor_match else None

    building_match = BUILDING_PATTERN.search(text_no_note)
    building = building_match.group(0).strip(" ,:-") if building_match else None

    street_match = STREET_PATTERN.search(text_no_note)
    street_address = street_match.group(0).strip(" ,:-") if street_match else None

    location_detail = text_no_note
    if ":" in text_no_note:
        _, location_detail = text_no_note.split(":", 1)
        location_detail = location_detail.strip()

    full_parts = [street_address or building or location_detail, ward, city]
    full_parsed = ", ".join([p for p in full_parts if p]) if any(full_parts) else None

    return {
        "city": city,
        "ward": ward,
        "district": district,
        "district_note": district_note,
        "floor": floor,
        "building": building,
        "street_address": street_address,
        "location_detail": location_detail if location_detail else None,
        "full_parsed": full_parsed,
        "parse_method": "solution_2"
    }

In [24]:
def clean_working_addresses_solution_3(raw_text, vietnam_cities):
    import re
    import unicodedata
    import pandas as pd

    DISTRICT_PATTERN = re.compile(
        r'(?i)\b(quận|huyện|thị xã|thành phố)\s+[^\n,;()]+'
    )

    FLOOR_PATTERN = re.compile(
        r'(?i)\b(?:tầng|lầu|floor|fl)\s*[a-z0-9,\-–/ ]+'
    )

    def normalize_empty_value(val):
        if pd.isna(val):
            return None
        val_str = str(val).strip()
        if not val_str or val_str.lower() in {"nan", "none", "null"}:
            return None
        return val_str

    def normalize_unicode(text):
        if text is None:
            return ""
        return unicodedata.normalize("NFC", str(text))

    def clean_address_text(text):
        text = normalize_empty_value(text)
        if text is None:
            return ""
        text = normalize_unicode(text)
        text = text.replace("\\n", "\n")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\s*\n\s*", "\n", text)
        return text.strip()

    def remove_leading_note(text):
        text = clean_address_text(text)
        if not text:
            return ""
        return re.sub(r'^\s*\([^)]*\)\s*', '', text).strip()

    def detect_city_from_text(text):
        text = clean_address_text(text)
        if not text:
            return None
        text_lower = text.lower()
        for city in vietnam_cities:
            city_norm = normalize_empty_value(city)
            if city_norm and city_norm.lower() in text_lower:
                return city_norm
        return None

    def extract_district_note(text):
        text = clean_address_text(text)
        if not text:
            return None
        matches = re.findall(r'\(([^)]*)\)', text)
        matches = [m.strip() for m in matches if normalize_empty_value(m)]
        return "; ".join(matches) if matches else None

    def remove_parentheses_content(text):
        text = clean_address_text(text)
        if not text:
            return ""
        return re.sub(r'\([^)]*\)', '', text).strip()

    def classify_chunk(chunk):
        chunk_lower = chunk.lower().strip()

        if re.search(r'\b(phường|xã|thị trấn)\b', chunk_lower):
            return "ward"

        if re.search(r'\b(tầng|lầu|floor|fl)\b', chunk_lower):
            return "floor_or_building"

        if re.search(r'\b(?:tòa|toà|tower|building|plaza|center|centre|complex|landmark|office)\b', chunk_lower):
            return "building"

        if re.search(r'\b(?:số\s*)?\d+[a-z0-9/\-]*\s+', chunk_lower):
            return "street"

        return "other"

    def parse_floor_and_building_from_chunk(chunk):
        floor_match = FLOOR_PATTERN.search(chunk)
        floor = floor_match.group(0).strip() if floor_match else None

        building = chunk
        if floor:
            building = FLOOR_PATTERN.sub("", building).strip(" ,-")
        if not building:
            building = None

        return floor, building

    def empty_result():
        return {
            "city": None,
            "ward": None,
            "district": None,
            "district_note": None,
            "floor": None,
            "building": None,
            "street_address": None,
            "location_detail": None,
            "full_parsed": None,
            "parse_method": "solution_3"
        }

    raw_text = normalize_empty_value(raw_text)
    if raw_text is None:
        return empty_result()

    text = clean_address_text(raw_text)
    text = remove_leading_note(text)
    text = re.sub(r'^\-\s*', '', text).strip()

    if not text:
        return empty_result()

    city = None
    detail = text

    if ":" in text:
        left, right = text.split(":", 1)
        detail = right.strip()
        city = detect_city_from_text(left.strip())

    if not city:
        city = detect_city_from_text(text)

    district_note = extract_district_note(detail)
    detail_no_note = remove_parentheses_content(detail)

    district_match = DISTRICT_PATTERN.search(detail_no_note)
    district = district_match.group(0).strip() if district_match else None

    chunks = [c.strip() for c in detail_no_note.split(",") if c.strip()]

    ward = None
    floor = None
    building = None
    street_address = None
    other_chunks = []

    for chunk in chunks:
        label = classify_chunk(chunk)

        if label == "ward" and ward is None:
            ward = chunk

        elif label == "floor_or_building":
            chunk_floor, chunk_building = parse_floor_and_building_from_chunk(chunk)
            if floor is None and chunk_floor:
                floor = chunk_floor
            if chunk_building:
                building = f"{building}, {chunk_building}" if building else chunk_building

        elif label == "building":
            building = f"{building}, {chunk}" if building else chunk

        elif label == "street" and street_address is None:
            street_address = chunk

        else:
            other_chunks.append(chunk)

    location_detail_parts = []
    if floor:
        location_detail_parts.append(floor)
    if building:
        location_detail_parts.append(building)
    if street_address:
        location_detail_parts.append(street_address)
    if other_chunks:
        location_detail_parts.extend(other_chunks)

    location_detail = ", ".join(location_detail_parts) if location_detail_parts else None

    full_parts = [street_address or building or location_detail, ward, city]
    full_parsed = ", ".join([p for p in full_parts if p]) if any(full_parts) else None

    return {
        "city": city,
        "ward": ward,
        "district": district,
        "district_note": district_note,
        "floor": floor,
        "building": building,
        "street_address": street_address,
        "location_detail": location_detail,
        "full_parsed": full_parsed,
        "parse_method": "solution_3"
    }

In [25]:
parsed_s1 = df["working_addresses_raw"].apply(
    lambda x: clean_working_addresses_solution_1(x, VIETNAM_CITIES)
)
df_s1 = pd.concat([df, pd.json_normalize(parsed_s1).add_prefix("s1_")], axis=1)

In [ ]:
parsed_s2 = df["working_addresses_raw"].apply(
    lambda x: clean_working_addresses_solution_2(x, VIETNAM_CITIES)
)
df_s2 = pd.concat([df, pd.json_normalize(parsed_s2).add_prefix("s2_")], axis=1)

In [32]:
parsed_s3 = df["working_addresses_raw"].apply(
    lambda x: clean_working_addresses_solution_3(x, VIETNAM_CITIES)
)
df_s3 = pd.concat([df, pd.json_normalize(parsed_s3).add_prefix("s3_")], axis=1)

In [33]:
df_s3.info()

<class 'pandas.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   job_url                  325 non-null    str  
 1   source_field_name        325 non-null    str  
 2   field_count              325 non-null    int64
 3   job_title_raw            325 non-null    str  
 4   salary_raw               325 non-null    str  
 5   location_raw             325 non-null    str  
 6   working_addresses_raw    325 non-null    str  
 7   working_times_raw        294 non-null    str  
 8   experience_raw           325 non-null    str  
 9   tags_raw                 325 non-null    str  
 10  job_level_raw            325 non-null    str  
 11  education_level_raw      325 non-null    str  
 12  employment_type_raw      325 non-null    str  
 13  job_quantity             325 non-null    str  
 14  deadline_raw             325 non-null    str  
 15  description_raw  

In [34]:
display(df_s3[["working_addresses_raw", "s3_full_parsed"]].head(10))

,working_addresses_raw,s3_full_parsed
0,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)","TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa, hồ chí minh"
1,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\nHà Nội:","61 Ngụy Như Kon Tum, Phường Thanh Xuân \nHà Nội:, hà nội"
2,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,3,5,6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (quận Thanh Xuân cũ)\nHà Nội:","6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Phường Thanh Xuân \nHà Nội:, hà nội"
3,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: FPT Tower, số 10 Phạm Văn Bạch, Phường Cầu Giấy (quận Cầu Giấy cũ)","số 10 Phạm Văn Bạch, Phường Cầu Giấy, hà nội"
4,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Landmark 72, Keangnam Phạm Hùng, Phường Yên Hòa (quận Cầu Giấy cũ)","Landmark 72, Phường Yên Hòa, hà nội"
5,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Keangnam Landmark 72, E6 Phạm Hùng, Phường Yên Hòa (quận Cầu Giấy cũ)","Keangnam Landmark 72, Phường Yên Hòa, hà nội"
6,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Keangnam Landmark 72, Phường Yên Hòa (quận Cầu Giấy cũ)","Keangnam Landmark 72, Phường Yên Hòa, hà nội"
7,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Khu Công nghệ cao Hòa Lạc, Xã Hòa Lạc (huyện Thạch Thất cũ) - Hà Nội: FPT Tower, số 10 Phạm Văn Bạch, Phường Cầu Giấy (quận Cầu Giấy cũ)\n- Hà Nội: Khu Công nghệ cao Hòa Lạc, Xã Hòa Lạc (huyện Thạch Thất cũ)\n- Hà Nội: FPT Tower, số 10 Phạm Văn Bạch, Phường Cầu Giấy (quận Cầu Giấy cũ)","số 10 Phạm Văn Bạch, Xã Hòa Lạc - Hà Nội: FPT Tower, hà nội"
8,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: 20 Cộng Hòa, Phường Bảy Hiền (quận Tân Bình cũ)","20 Cộng Hòa, Phường Bảy Hiền, hồ chí minh"
9,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: 20 Cộng Hòa, Phường Bảy Hiền (quận Tân Bình cũ)","20 Cộng Hòa, Phường Bảy Hiền, hồ chí minh"


In [ ]:
display(df_s3[["working_addresses_raw", "s3_full_parsed"]].head(10))

,working_addresses_raw,s3_full_parsed
0,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)","TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa, hồ chí minh"
1,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\nHà Nội:","61 Ngụy Như Kon Tum, Phường Thanh Xuân \nHà Nội:, hà nội"
2,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,3,5,6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (quận Thanh Xuân cũ)\nHà Nội:","6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Phường Thanh Xuân \nHà Nội:, hà nội"
3,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: FPT Tower, số 10 Phạm Văn Bạch, Phường Cầu Giấy (quận Cầu Giấy cũ)","số 10 Phạm Văn Bạch, Phường Cầu Giấy, hà nội"
4,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Landmark 72, Keangnam Phạm Hùng, Phường Yên Hòa (quận Cầu Giấy cũ)","Landmark 72, Phường Yên Hòa, hà nội"
5,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Keangnam Landmark 72, E6 Phạm Hùng, Phường Yên Hòa (quận Cầu Giấy cũ)","Keangnam Landmark 72, Phường Yên Hòa, hà nội"
6,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Keangnam Landmark 72, Phường Yên Hòa (quận Cầu Giấy cũ)","Keangnam Landmark 72, Phường Yên Hòa, hà nội"
7,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Khu Công nghệ cao Hòa Lạc, Xã Hòa Lạc (huyện Thạch Thất cũ) - Hà Nội: FPT Tower, số 10 Phạm Văn Bạch, Phường Cầu Giấy (quận Cầu Giấy cũ)\n- Hà Nội: Khu Công nghệ cao Hòa Lạc, Xã Hòa Lạc (huyện Thạch Thất cũ)\n- Hà Nội: FPT Tower, số 10 Phạm Văn Bạch, Phường Cầu Giấy (quận Cầu Giấy cũ)","số 10 Phạm Văn Bạch, Xã Hòa Lạc - Hà Nội: FPT Tower, hà nội"
8,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: 20 Cộng Hòa, Phường Bảy Hiền (quận Tân Bình cũ)","20 Cộng Hòa, Phường Bảy Hiền, hồ chí minh"
9,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: 20 Cộng Hòa, Phường Bảy Hiền (quận Tân Bình cũ)","20 Cộng Hòa, Phường Bảy Hiền, hồ chí minh"


In [19]:
def clean_working_addresses(raw_text):
    if pd.isna(raw_text) or not str(raw_text).strip():
        return []

    text = str(raw_text).strip()

    # 1) bỏ note đầu chuỗi trong ngoặc nếu có
    text = re.sub(r'^\s*\([^)]*\)\s*', '', text).strip()

    # 2) chuẩn hóa xuống dòng
    text = text.replace('\\n', '\n')

    # 3) tách từng dòng có địa chỉ
    lines = [line.strip() for line in text.split('\n') if line.strip()]

    results = []

    for line in lines:
        # bỏ dấu đầu dòng "-"
        line = re.sub(r'^\-\s*', '', line).strip()

        # 4) tách city theo dấu :
        city = None
        detail = line

        if ':' in line:
            left, right = line.split(':', 1)
            city = left.strip()
            detail = right.strip()

        # 5) lấy note trong ngoặc ở cuối
        note_matches = re.findall(r'\(([^)]*)\)', detail)
        district_note = '; '.join(note_matches) if note_matches else None

        # bỏ phần ngoặc ra khỏi detail
        detail_clean = re.sub(r'\([^)]*\)', '', detail).strip()

        # 6) split theo dấu phẩy
        parts = [p.strip() for p in detail_clean.split(',') if p.strip()]

        street = None
        ward = None
        district = None

        if len(parts) >= 1:
            street = parts[0]

        for p in parts[1:]:
            p_lower = p.lower()
            if p_lower.startswith(('tầng', 'floor', 'tòa')):
                location_detail = p
            elif p_lower.startswith(('phường', 'xã', 'thị trấn')):
                ward = p
            elif p_lower.startswith(('quận', 'huyện', 'thị xã', 'thành phố')):
                district = p
            elif street is None:
                street = p

        results.append({
            'city': city,
            'street': street,
            'ward': ward,
            'location_detail': location_detail if 'location_detail' in locals() else None,
            'district': district,
            'district_note': district_note,
            'full_parsed': f"{street}, {ward}, {city}" if street and ward and city else None
        })

    return results

In [20]:
df_clean["working_addresses_clean"] = df_clean["working_addresses_raw"].apply(clean_working_addresses)

In [13]:
def clean_working_addresses(text: str) -> str:
    """
    Clean working_addresses_raw thành địa chỉ hoàn chỉnh, dễ đọc.
    """
    if pd.isna(text) or not str(text).strip():
        return ""
    
    text = str(text).strip()
    
    # Bước 1: Loại bỏ phần mở đầu dài dòng (phần trong ngoặc đầu tiên)
    text = re.sub(r'^\s*\(đã được cập nhật theo danh mục hành chính mới.*?\)\s*[-–—]\s*', '', text, flags=re.IGNORECASE)
    
    # Bước 2: Loại bỏ các cụm ghi chú thừa còn lại trong ngoặc
    text = re.sub(r'\s*\([^)]*quận/huyện cũ tương ứng.*?\)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*\([^)]*dễ dàng tra cứu.*?\)', '', text, flags=re.IGNORECASE)
    
    # Bước 3: Loại bỏ tất cả nội dung trong ngoặc đơn (an toàn)
    text = re.sub(r'\([^)]*\)', '', text)
    
    # Bước 4: Chuẩn hóa dấu phân cách (thay - , ; bằng dấu phẩy)
    text = re.sub(r'\s*[-–—;,]\s*', ', ', text)
    
    # Bước 5: Sử dụng clean_text để lowercase + chuẩn hóa khoảng trắng
    text = clean_text(text)
    
    # Bước 6: Viết hoa chữ cái đầu mỗi thành phần (tùy chọn nhưng nên làm cho đẹp)
    def capitalize_parts(s):
        parts = [p.strip() for p in s.split(',') if p.strip()]
        capitalized = [p.capitalize() if not p[0].isdigit() else p for p in parts]
        return ', '.join(capitalized)
    
    text = capitalize_parts(text)
    
    return text


# ==================== ÁP DỤNG ====================
df_clean["working_addresses_clean"] = df_clean["working_addresses_raw"].apply(clean_working_addresses)

# Kiểm tra kết quả
print("Kết quả sau khi clean working_addresses:")
display(df_clean[["working_addresses_raw", "working_addresses_clean"]].head(1))

Kết quả sau khi clean working_addresses:


,working_addresses_raw,working_addresses_clean
0,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)","Hồ chí minh ttc 253 hoàng văn thụ, Phường tân sơn hòa"


In [ ]:
df_clean[["job_title_raw", "working_addresses_raw", "working_addresses_clean"]].head(10)

- Chuẩn hóa lương (salary)

In [ ]:
def parse_salary(text: str):
    text = clean_text(text)
    if not text:
        return None, None, "unknown"
    if any(x in text for x in ["thỏa thuận", "thuong luong", "cạnh tranh"]):
        return None, None, "negotiable"
    
    nums = re.findall(r"\d+(?:[.,]\d+)?", text)
    nums = [float(n.replace(",", ".")) for n in nums]
    
    if "triệu" in text or "trieu" in text:
        nums = [n * 1_000_000 for n in nums]
    elif "usd" in text:
        pass  # giữ nguyên
    
    nums = [int(round(n)) for n in nums]
    
    if len(nums) >= 2:
        return nums[0], nums[1], "range"
    if len(nums) == 1:
        return nums[0], nums[0], "fixed"
    return None, None, "unknown"


def parse_experience(text: str):
    text = clean_text(text)
    if not text:
        return None, None, "unknown"
    if "không yêu cầu" in text:
        return 0, 0, "no_requirement"
    if any(x in text for x in ["thực tập", "intern", "fresher"]):
        return 0, 1, "intern_fresher"
    
    nums = [int(x) for x in re.findall(r"\d+", text)]
    if len(nums) >= 2:
        return nums[0], nums[1], "range"
    if len(nums) == 1:
        if any(x in text for x in ["+", "trên", "ít nhất"]):
            return nums[0], None, "min_only"
        return nums[0], nums[0], "fixed"
    return None, None, "unknown"


salary_parsed = df["salary_raw"].apply(parse_salary)
df["salary_min"]  = [x[0] for x in salary_parsed]
df["salary_max"]  = [x[1] for x in salary_parsed]
df["salary_type"] = [x[2] for x in salary_parsed]

exp_parsed = df["experience_raw"].apply(parse_experience)
df["experience_min_years"] = [x[0] for x in exp_parsed]
df["experience_max_years"] = [x[1] for x in exp_parsed]
df["experience_type"]      = [x[2] for x in exp_parsed]


# Kiểm tra
print("\nSalary types:\n", df["salary_type"].value_counts())
print("\nExperience types:\n", df["experience_type"].value_counts())

In [ ]:
def extract_skills(text: str, skill_dict: dict) -> list:
    text = clean_text(text)
    found = set()
    for canonical, aliases in skill_dict.items():
        for alias in aliases:
            alias_clean = clean_text(alias)
            pattern = rf"(?<!\w){re.escape(alias_clean)}(?!\w)"
            if re.search(pattern, text, re.IGNORECASE):
                found.add(canonical)
                break
    return sorted(found)


# Nguồn text để extract skill
skill_source = (
    df["job_title_clean"].fillna("") + " " +
    df["requirements_clean"].fillna("") + " " +
    df["description_clean"].fillna("")
)

df["skills_normalized"] = skill_source.apply(lambda x: extract_skills(x, SKILL_DICT))
df["skills_normalized_str"] = df["skills_normalized"].apply(lambda x: " | ".join(x) if x else "")

# Kiểm tra
print("Top 15 skill phổ biến:")
skill_flat = [s for sublist in df["skills_normalized"] for s in sublist]
pd.Series(skill_flat).value_counts().head(15)

In [ ]:
def build_job_text(row):
    parts = []
    if row.get("job_title_clean"):
        parts.append(f"[title] {row['job_title_clean']}")
    if row.get("requirements_clean"):
        parts.append(f"[requirements] {row['requirements_clean']}")
    if row.get("description_clean"):
        parts.append(f"[description] {row['description_clean']}")
    if row.get("skills_normalized"):
        parts.append(f"[skills] {' | '.join(row['skills_normalized'])}")
    return " ".join(parts).strip()


df["job_text"] = df.apply(build_job_text, axis=1)

# Kiểm tra mẫu
print("Ví dụ job_text:")
print(df["job_text"].iloc[0][:600] + "..." if len(df["job_text"].iloc[0]) > 600 else df["job_text"].iloc[0])

In [ ]:
def select_final_columns(df):
    final_cols = [
        "job_url", "source_field_name", "field_count",
        "job_title_raw", "job_title_clean",
        "company_name_raw", "company_url",
        "salary_raw", "salary_min", "salary_max", "salary_type",
        "location_raw", "location_normalized",
        "experience_raw", "experience_min_years", "experience_max_years", "experience_type",
        "job_level_raw", "education_level_raw", "employment_type_raw", "deadline_raw",
        "company_scale_raw", "company_address_raw", "company_description_raw",
        "description_raw", "description_clean",
        "requirements_raw", "requirements_clean",
        "benefits_raw", "benefits_clean",
        "skills_normalized", "skills_normalized_str",
        "job_text"
    ]
    existing = [c for c in final_cols if c in df.columns]
    return df[existing].copy()


final_df = select_final_columns(df)
print("Final shape:", final_df.shape)
display(final_df.head(3))

In [ ]:
def save_processed_data(df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    csv_path = os.path.join(output_dir, f"jobs_nlp_ready_{ts}.csv")
    xlsx_path = os.path.join(output_dir, f"jobs_nlp_ready_{ts}.xlsx")
    
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved CSV → {csv_path}")
    
    try:
        df.to_excel(xlsx_path, index=False)
        print(f"Saved XLSX → {xlsx_path}")
    except Exception as e:
        print(f"Không lưu được Excel: {e}")


# Lưu
save_processed_data(final_df, OUTPUT_DIR)

In [ ]:
# Export nhanh để test (không timestamp)
quick_path = os.path.join(OUTPUT_DIR, "jobs_nlp_ready_latest_debug.csv")
final_df.to_csv(quick_path, index=False, encoding="utf-8-sig")
print("Quick debug file:", quick_path)

In [ ]:
df_clean.info()

In [ ]:
df_final = pd.DataFrame()
df_final["job_url"] = df_clean["job_url"]
df_final["job_title"] = df_clean["job_title_raw"]
df_final["tags"] = df_clean["tags_raw"]
# df_final["salary"] = df_clean["salary_clean"]
df_final["location"] = df_clean["location_clean"]

# df_final["working_addresses"] = df_clean["working_addresses_clean"]
# df_final["working_times"] = df_clean["working_times_clean"]
# df_final["experience"] = df_clean["experience_clean"]

# df_final["job_level"] = df_clean["job_level_clean"]
# df_final["education_level"] = df_clean["education_level_clean"]
# df_final["employment_type"] = df_clean["employment_type_clean"]
# df_final["job_quantity"] = df_clean["job_quantity_clean"]

# df_final["deadline"] = df_clean["deadline_clean"]

# df_final["description"] = df_clean["description_clean"]
# df_final["requirements"] = df_clean["requirements_clean"]
# df_final["benefits"] = df_clean["benefits_clean"]

# df_final["company_name"] = df_clean["company_name_clean"]
# df_final["company_url"] = df_clean["company_url_clean"]
# df_final["company_website"] = df_clean["company_website_clean"]
# df_final["company_scale"] = df_clean["company_scale_clean"]
# df_final["company_field"] = df_clean["company_field_clean"]
# df_final["company_address"] = df_clean["company_address_clean"]
# df_final["company_description"] = df_clean["company_description_clean"]


In [ ]:
df_final.head()